# A custom aggregator

This notebook defines an aggregator and trains FedNCF with it. The rule is a toy: at each coordinate, keep the value uploaded by one client drawn uniformly at random. The draw is independent for every coordinate.

Any method can use the class. FedNCF is the server in this run. The table is synthetic and the run is two rounds, so it finishes on a CPU. `examples/quickstart.ipynb` is the 100-round MovieLens configuration. `examples/custom_data.ipynb` covers loading a table you already have.

FedNCF (Perifanis and Efraimidis, *Knowledge-Based Systems*, 2022) is neural collaborative filtering with one user per client. The server holds the item embeddings and the shared network. Each client holds that user's embeddings.

## Install

FedRecLab is installed from GitHub. The command also installs NumPy, PyTorch, pandas, tqdm and Cornac. PyTorch is a large download. Run this cell before any import. The notebook does not use the repository working directory, so the same cell works in Colab from a fresh environment.

In [ ]:
%pip install "git+https://github.com/aymenkhs/FedRecLab.git"

## What an aggregator implements

Subclass `BaseAggregation` and implement `aggregate`. The server calls it once per dense parameter group, after the selected clients have uploaded that group. The arguments are:

- `round_info` records who was selected, who responded, how many training examples each client had, and their losses. `FedAvg` reads `num_training_examples` from here.
- `previous_parameters` is the value before this round. `FedAtt` reads it.
- `updates` holds one value per client who sent the group. A dense numeric group arrives as one stacked array or tensor, and the first axis is the clients. A dict group, such as a state dict, arrives as a list of dicts. Compute in NumPy. Return the same type, dtype and device as the input: `_to_torch_like` on the base class puts a NumPy result back onto a tensor.
- `param_io` is the group's declaration (`residency`, `exchange`, `apply_to`, `layout`). The method decides whether the uploads are parameters, deltas or gradients. The aggregator combines those uploads. The server applies the combined result.

Return one value in the client's layout: one array, or one dict with the same keys. Raise on an empty upload, on dict keys that differ across clients, and on a non-finite value.

The constructor stores `num_clients`, the population size. The clients inside one `aggregate` call are the ones who sent that group, so there can be fewer of them.

Pass aggregators to the server as `aggregation_strategies`: one aggregator applied to every group, or a dict from group name to aggregator. Build a new aggregator for every server, and one instance per group. `on_round_start` and `on_round_end` are there when a rule needs the previous round or a note after the round. The class below leaves both empty.

## Random coordinate selection

`RandomCoordinate` draws a client index for every coordinate with a `numpy.random.Generator` created from `seed`. The generator advances on every `aggregate` call, including once per row when the group is row layout. The same seed and the same sequence of calls reproduce the choices. The draws stay on this generator. They do not touch the global NumPy or PyTorch RNGs.

Dict keys are reduced in sorted order, so the sequence of draws follows the key names. Two clients who built their dicts with different insertion orders still match.

A row-layout group uploads `{rows, values}` for the item rows that client trained. The base-class `aggregate_rows` is enough for that layout: it calls `aggregate` once per uploaded row, and only with the clients who sent the row. Override `aggregate_rows` when you want a vectorised reduce over the packed `(n, k, d)` array. `Median` and `FedAvg` do that. This toy uses the base-class loop.

The rule reads every client's upload. A sum-only secure-aggregation payload does not carry the per-client values this choice needs.

In [ ]:
import numpy as np
import torch

from fedreclab.federated.aggregation import BaseAggregation


class RandomCoordinate(BaseAggregation):
    """Keep, at each coordinate, the value of one randomly chosen client.

    The client index is drawn independently and uniformly for every
    coordinate. Dict groups are reduced one key at a time, in sorted key
    order. Draws come from this instance's generator, so the same seed
    and the same sequence of calls reproduce the same choices.

    ``updates`` holds one value per client who sent this group. On a
    row-layout group the base class calls ``aggregate`` once per
    uploaded row, with only the clients who sent that row.
    ``round_info``, ``previous_parameters`` and ``param_io`` are unread.

    Requires the individual client updates. Their sum does not determine
    the coordinate-wise choice.

    Parameters
    ----------
    num_clients : int
        Population size stored on ``BaseAggregation``. The draw is over
        the updates in this call, which can be fewer than ``num_clients``.
    seed : int
        Seed for the NumPy generator that draws client indices.

    Raises
    ------
    ValueError
        If ``updates`` is empty, dict keys differ across clients, or any
        value is non-finite.
    TypeError
        If ``updates`` is neither a stacked array nor a list of updates.

    Examples
    --------
    With two client vectors ``[10, 20]`` and ``[11, 21]`` and a fixed
    seed, each coordinate of the result equals that coordinate of one
    of the two vectors. A second instance with the same seed returns
    the same vector.
    """

    def __init__(self, num_clients: int, seed: int):
        super().__init__(num_clients)
        self._rng = np.random.default_rng(seed)

    def aggregate(self, round_info, previous_parameters, updates, param_io):
        """Pick one client's value at each coordinate.

        Parameters
        ----------
        round_info : RoundStatistics
            Unused. Present for the aggregator interface.
        previous_parameters : array or dict of arrays
            Unused.
        updates : stacked array or list
            Per-client tensors, arrays, or dicts of those. A stacked
            array's first axis is the clients.
        param_io : ParamIO
            Group declaration. Unused.

        Returns
        -------
        array, Tensor, or dict
            One value per coordinate, same layout as a single client.
            A tensor result matches the first client's dtype and device.
        """
        clients = self._clients(updates)
        if isinstance(clients[0], dict):
            keys = self._keys(clients)
            return {
                key: self._pick([client[key] for client in clients])
                for key in keys
            }
        return self._pick(clients)

    def _clients(self, updates):
        if isinstance(updates, (torch.Tensor, np.ndarray)):
            if int(updates.shape[0]) == 0:
                raise ValueError("No client updates provided")
            return [updates[i] for i in range(int(updates.shape[0]))]
        if isinstance(updates, list):
            if len(updates) == 0:
                raise ValueError("No client updates provided")
            return updates
        raise TypeError(
            "Unsupported updates type: "
            f"{type(updates).__name__}. Pass a stacked array or a list "
            "of client updates."
        )

    @staticmethod
    def _keys(clients):
        keys = set(clients[0])
        for index, client in enumerate(clients[1:], start=1):
            if set(client) != keys:
                raise ValueError(
                    f"Client update dict keys mismatch at index {index}"
                )
        return sorted(keys)

    @staticmethod
    def _numpy(value):
        if isinstance(value, torch.Tensor):
            return value.detach().cpu().numpy()
        return np.asarray(value)

    def _pick(self, clients):
        stacked = np.stack([self._numpy(client) for client in clients], axis=0)
        if not np.isfinite(stacked).all():
            raise ValueError(
                "Non-finite values in RandomCoordinate client updates"
            )
        n_clients = stacked.shape[0]
        choice = self._rng.integers(0, n_clients, size=stacked.shape[1:])
        # Index shape (1, *parameter_shape) selects along the client axis.
        picked = np.take_along_axis(
            stacked, np.expand_dims(choice, axis=0), axis=0
        )[0].copy()
        first = clients[0]
        if isinstance(first, torch.Tensor):
            return self._to_torch_like(picked, first)
        return picked

## Check a dense group

These updates never go through a server. `RoundStatistics` and `ParamIO` are the objects the server would pass. `RandomCoordinate` leaves them unread, so the placeholders below only satisfy the signature.

The first list is three length-4 vectors. A second generator, seeded the same way, replays the client index of each coordinate. Index 0 is the first client in the list, which is the order the server stacks participants. The same seed returns the same vector. A different seed returns another one.

The tensor check stacks two clients on axis 0, the shape a dense numeric group has inside `aggregate`. The result stays a `float32` CPU tensor.

The dict check builds the two clients with opposite key order. Sorted keys draw `b` before `w`. A non-finite value and an empty upload both raise.

In [ ]:
from fedreclab.federated.utils import ParamIO, RoundStatistics

SEED = 42


def round_info(client_ids):
    """Placeholder round record. RandomCoordinate does not read it."""
    client_ids = list(client_ids)
    return RoundStatistics(
        current_round=0,
        selected=client_ids,
        responded=client_ids,
        dropped=[],
        num_training_examples={client_id: 1 for client_id in client_ids},
        training_losses={client_id: [] for client_id in client_ids},
        training_durations={client_id: 0.0 for client_id in client_ids},
    )


param_io = ParamIO(residency="server", exchange="params", apply_to="server")

clients = [
    np.array([10.0, 20.0, 30.0, 40.0]),
    np.array([11.0, 21.0, 31.0, 41.0]),
    np.array([12.0, 22.0, 32.0, 42.0]),
]
result = RandomCoordinate(num_clients=3, seed=SEED).aggregate(
    round_info([0, 1, 2]), None, clients, param_io
)
twin = np.random.default_rng(SEED)
choice = twin.integers(0, len(clients), size=clients[0].shape)
expected = np.stack(clients, axis=0)[choice, np.arange(clients[0].size)]
print("chosen client per coordinate:", choice)
print("aggregated:", result)
assert np.array_equal(result, expected)

again = RandomCoordinate(num_clients=3, seed=SEED).aggregate(
    round_info([0, 1, 2]), None, clients, param_io
)
other = RandomCoordinate(num_clients=3, seed=SEED + 1).aggregate(
    round_info([0, 1, 2]), None, clients, param_io
)
assert np.array_equal(result, again)
assert not np.array_equal(result, other)

stacked = torch.tensor(
    [[10.0, 20.0, 30.0], [11.0, 21.0, 31.0]], dtype=torch.float32
)
torch_result = RandomCoordinate(num_clients=2, seed=SEED).aggregate(
    round_info([0, 1]), None, stacked, param_io
)
torch_choice = np.random.default_rng(SEED).integers(0, 2, size=(3,))
torch_expected = stacked.detach().cpu().numpy()[torch_choice, np.arange(3)]
print("torch choice:", torch_choice, "result:", torch_result)
assert isinstance(torch_result, torch.Tensor)
assert torch_result.dtype == torch.float32
assert np.array_equal(torch_result.detach().cpu().numpy(), torch_expected)

dict_updates = [
    {"w": np.array([1.0, 2.0]), "b": np.array([3.0])},
    {"b": np.array([4.0]), "w": np.array([5.0, 6.0])},
]
dict_result = RandomCoordinate(num_clients=2, seed=SEED).aggregate(
    round_info([0, 1]), None, dict_updates, param_io
)
dict_twin = np.random.default_rng(SEED)
b_choice = dict_twin.integers(0, 2, size=(1,))
w_choice = dict_twin.integers(0, 2, size=(2,))
b_stack = np.stack([update["b"] for update in dict_updates], axis=0)
w_stack = np.stack([update["w"] for update in dict_updates], axis=0)
print("b choice:", b_choice, "value:", dict_result["b"])
print("w choice:", w_choice, "value:", dict_result["w"])
assert np.array_equal(dict_result["b"], b_stack[b_choice, np.arange(1)])
assert np.array_equal(dict_result["w"], w_stack[w_choice, np.arange(2)])

try:
    RandomCoordinate(num_clients=2, seed=SEED).aggregate(
        round_info([0, 1]),
        None,
        [np.array([1.0]), np.array([np.nan])],
        param_io,
    )
except ValueError as exc:
    print("non-finite:", exc)
else:
    raise AssertionError("non-finite updates should fail")

try:
    RandomCoordinate(num_clients=2, seed=SEED).aggregate(
        round_info([0, 1]), None, [], param_io
    )
except ValueError as exc:
    print("empty:", exc)
else:
    raise AssertionError("empty updates should fail")

## Check a row-layout group

`aggregate_rows` receives the union of rows uploaded this round. `values` has shape `(n, k, d)`: clients, union rows, coordinates. `mask[i, j]` is true when client `i` sent union row `j`. `previous_rows` has shape `(k, d)`. This aggregator ignores it.

Here one row was sent by clients 0 and 1. Client 2's values are 3 and 30, and its mask entry is false, so neither coordinate can be 3 or 30. The base class calls `aggregate` on the two senders only. With `seed=42` the two coordinates come from different senders.

In [ ]:
values = np.array(
    [[[1.0, 10.0]], [[2.0, 20.0]], [[3.0, 30.0]]], dtype=np.float64
)
mask = np.array([[True], [True], [False]])
previous_rows = np.zeros((1, 2), dtype=np.float64)

row_result = RandomCoordinate(num_clients=3, seed=SEED).aggregate_rows(
    round_info([0, 1, 2]), previous_rows, values, mask, param_io
)
print(row_result)
assert row_result.shape == (1, 2)
assert row_result[0, 0] in (1.0, 2.0)
assert row_result[0, 1] in (10.0, 20.0)
assert np.allclose(row_result, [[1.0, 20.0]])

## A small dataset

`load_dataframe_dataset` reads a pandas DataFrame and returns an `InteractionDataset`. That object is the only copy of the interactions. It remaps user and item ids to contiguous integers starting at 0, splits, and assigns clients. The frame needs `user_id` and `item_id`. `rating` and `timestamp` are optional. A chronological split reads `timestamp`.

The frame below has 12 users, 20 items and 8 distinct items per user. Ratings are integers from 1 to 5. Timestamps increase within each user. The split is stratified by user and chronological: 10% validation, 20% test. Cornac turns each fraction into a count with `ceil`, separately for every user. `ceil(0.1 * 8) + ceil(0.2 * 8)` is 3, so five interactions per user stay in training. `CrossDevicePartitionStrategy` assigns each training user to their own client. `seed=42` is `SEED` from the check above.

In [ ]:
import pandas as pd

from fedreclab.data.client_partitioning import CrossDevicePartitionStrategy
from fedreclab.data.load_data import load_dataframe_dataset
from fedreclab.data.split import StratifiedSplitStrategy

rng = np.random.default_rng(SEED)
n_users = 12
n_items = 20
per_user = 8

user_ids = np.repeat(np.arange(n_users), per_user)
item_pool = np.arange(n_items)
item_ids = np.concatenate(
    [rng.choice(item_pool, size=per_user, replace=False) for _ in range(n_users)]
)
ratings = rng.integers(1, 6, size=user_ids.shape[0]).astype(float)
base = rng.integers(1_600_000_000, 1_700_000_000, size=n_users)
offsets = np.sort(rng.integers(0, 10_000_000, size=(n_users, per_user)), axis=1)
timestamps = (base[:, None] + offsets).ravel()

interactions = pd.DataFrame(
    {
        "user_id": user_ids,
        "item_id": item_ids,
        "rating": ratings,
        "timestamp": timestamps,
    }
)

dataset = load_dataframe_dataset(
    interactions,
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True,
        include_test=True,
        val_size=0.1,
        test_size=0.2,
        chrono=True,
        by="user",
        seed=SEED,
    ),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True,
    include_test=True,
    seed=SEED,
    k_core=0,
    min_user_interactions=0,
    min_item_interactions=0,
)

print(
    f"{dataset.num_users} users, {dataset.num_items} items, "
    f"{dataset.num_interactions} interactions, {dataset.num_clients} clients"
)
dataset.interaction_data["split"].value_counts()

## Train

Each group names its own aggregator, so one group can use `RandomCoordinate` while another keeps `FedAvg`. This run uses `RandomCoordinate` for all three FedNCF groups. Each name gets its own instance and therefore its own generator. A single shared instance would spend one stream of draws across every group.

`item_embeddings` is row layout and a dict of two item tables. Each round calls `aggregate` once per uploaded item row, and the dict path draws coordinates of each table separately. Clients upload those rows in the clear, so the server sees which items each selected client updated. `shared_layers` and `output_weights` are dense dicts: one `aggregate` call per group per round.

`FedNCFClientAlgorithm` runs local training. One algorithm object is shared across clients. Each client's user embeddings live in a client state that the server creates. `predictive_factor` is the latent width and is the same value on the client and the server. Observed interactions are implicit positives, four unobserved items are sampled as negatives, and the loss is binary cross-entropy. One local epoch keeps the demo short.

`RandomClientSelection` draws half of the 12 clients each round. `seed=42` on the server seeds PyTorch and, because the selector is built without its own seed, the client draws as well. The aggregator seeds are separate from that server seed.

The server is built without an evaluator, so the two rounds do no validation. Client algorithms, selectors and aggregators are single-use. A second server needs new ones. The dataset may be shared.

In [ ]:
from fedreclab.federated.client_selection import RandomClientSelection
from fedreclab.methods.fedncf.fedncf import FedNCF
from fedreclab.methods.fedncf.fedncf_client import FedNCFClientAlgorithm

client = FedNCFClientAlgorithm(
    predictive_factor=4,
    local_epochs=1,
    lr=5e-4,
    batch_size=32,
    neg_samples=4,
)
server = FedNCF(
    predictive_factor=4,
    interaction_dataset=dataset,
    client_algorithm=client,
    num_rounds=2,
    seed=SEED,
    client_selection_strategy=RandomClientSelection(
        dataset.num_clients, 0.5, "frac"
    ),
    aggregation_strategies={
        "item_embeddings": RandomCoordinate(
            num_clients=dataset.num_clients, seed=SEED
        ),
        "shared_layers": RandomCoordinate(
            num_clients=dataset.num_clients, seed=SEED
        ),
        "output_weights": RandomCoordinate(
            num_clients=dataset.num_clients, seed=SEED
        ),
    },
    evaluator=None,
)
server.run()

## Evaluate

`FullRankingEvaluator` scores, for each user, every catalogue item except items from the excluded splits, then averages HR and NDCG at 10. Test candidates drop that user's training and validation items. Scoring uses the client algorithm passed in, so this evaluator and the server share the algorithm from the training cell. Build a new evaluator for a new server.

The score uses the parameters training finished with: `global_parameters`, `client_states` and `extra_payload`. `current_round` is the last completed round. The result is a dict with keys `ndcg@10` and `hr@10`. Each value is the mean over users who have test interactions. On 20 items, HR@10 is a large fraction of the catalogue, so the number is only a check that the run produced a score.

In [ ]:
from fedreclab.data.evaluator import FullRankingEvaluator

evaluator = FullRankingEvaluator(
    metrics=["ndcg", "hr"],
    topk=10,
    interaction_dataset=dataset,
    client_algorithm=client,
)
print(evaluator.evaluate(
    current_round=server.current_round,
    global_parameters=server.global_parameters,
    client_states=server.client_states,
    extra_payload=server.extra_payload,
    split="test",
))